In [ ]:
# es/data-analysis/normal/06-missing-values
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("titanic.csv", ())


## Por qué importan los valores faltantes

Casi todos los conjuntos de datos reales tienen valores faltantes. Si los ignoras, las agregaciones devuelven NaN, las visualizaciones se rompen y los modelos de aprendizaje automático fallan. El primer paso en cualquier análisis es comprender y abordar los datos faltantes.


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")


## Detectando valores faltantes

**Verifica una sola columna:**


In [ ]:
print(df["Age"].isna().sum())   # 177 missing Age values


**Verifica todas las columnas a la vez:**


In [ ]:
print(df.isna().sum())


Salida:


In [ ]:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


**Mira el porcentaje faltante:**


In [ ]:
print((df.isna().sum() / len(df) * 100).round(1))


Salida:


In [ ]:
Cabin          77.1%
Age            19.9%
Embarked        0.2%
...


Cabin falta en un 77% — demasiado para rellenarlo de forma significativa. Age falta en un 20% — vale la pena intentar rellenarla. Embarked tiene solo 2 faltantes — fácil de manejar.

## Eliminando valores faltantes

**Elimina filas con cualquier valor faltante:**


In [ ]:
df_clean = df.dropna()
print(df_clean.shape)   # (183, 12) — lost most rows


Esto es demasiado agresivo para la mayoría de los conjuntos de datos. Pierdes 708 de 891 filas.

**Elimina filas donde todos los valores faltan:**


In [ ]:
df_clean = df.dropna(how="all")


**Elimina filas que faltan en columnas específicas:**


In [ ]:
df_clean = df.dropna(subset=["Age", "Embarked"])
print(df_clean.shape)   # (712, 12) — much better


**Elimina columnas con demasiados valores faltantes:**


In [ ]:
# Drop columns where more than 50% is missing
threshold = len(df) * 0.5
df_clean = df.dropna(thresh=threshold, axis=1)


## Rellenando valores faltantes

**Rellena con una constante:**


In [ ]:
df["Embarked"] = df["Embarked"].fillna("S")   # most common port


**Rellena con una estadística:**


In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())


**Rellenar hacia adelante o hacia atrás** — útil para series de tiempo:


In [ ]:
# Use the previous valid value to fill gaps
df["Price"] = df["Price"].ffill()

# Use the next valid value
df["Price"] = df["Price"].bfill()


**Rellena con valores diferentes por columna:**


In [ ]:
fill_values = {"Age": df["Age"].median(), "Embarked": "S", "Cabin": "Unknown"}
df = df.fillna(fill_values)


## Elegir una estrategia

| Escenario | Estrategia |
|---|---|
| Los valores faltantes son aleatorios y pocos (< 5%) | Eliminar con `dropna(subset=[...])` |
| Valores faltantes en una columna numérica | Rellenar con la mediana (robusta a los atípicos) |
| Valores faltantes en una columna categórica | Rellenar con la moda o "Desconocido" |
| La columna falta en > 50% | Eliminar la columna completa |
| Datos de series de tiempo | Usar `ffill()` o `bfill()` |

## Errores comunes

**Rellenar antes de dividir en entrenamiento/prueba** — esto filtra información. Calcula los valores de relleno solo con los datos de entrenamiento y luego aplícalos a ambos.

**Eliminar demasiado agresivamente** — verifica siempre cuántas filas pierdes. `dropna()` sin argumentos suele eliminar mucho más de lo esperado.

**Olvidar verificar** — ejecuta siempre `df.isna().sum()` después de rellenar para confirmar que no queden valores NaN.

## Inténtalo

Del conjunto de datos del Titanic:
1. Calcula el porcentaje de valores faltantes para cada columna
2. Elimina la columna Cabin (demasiados valores faltantes)
3. Rellena Age con la edad mediana
4. Rellena Embarked con el valor más común
5. Verifica que no queden valores faltantes


In [ ]:
import pandas as pd

# titanic.csv ships with the course — load it from the browser file system.
df = pd.read_csv("titanic.csv")

print((df.isna().sum() / len(df) * 100).round(1))

df = df.drop(columns=["Cabin"])
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isna().sum())


## Conclusiones clave

- Inspecciona siempre los valores faltantes primero con `isna().sum()` antes de decidir una estrategia
- `dropna()` es potente, pero suele ser demasiado agresivo sin `subset` o `thresh`
- `fillna()` con la mediana o la moda es la estrategia de relleno más común
- Las columnas con > 50% de valores faltantes generalmente es mejor eliminarlas que rellenarlas

## Desafío de práctica

Carga el conjunto de datos del Titanic y crea una versión limpia: elimina Cabin, rellena Age con la mediana, rellena Embarked con la moda. Luego compara la tasa de supervivencia antes y después de la limpieza. ¿La limpieza cambió la tasa de supervivencia general? ¿Por qué sí o por qué no?


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
